## 실습 - 아래 데이터로 RAG 구축
- 16-1_K희망사다리2026_모두의정책.pdf
- 16-5_Samsung_Electronics_Sustainability_Report_2026_KOR.pdf
- 16-6_개인정보FAQ.csv

### 1. K 희망사다리 pdf

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()

True

In [2]:
model = init_chat_model("openai:gpt-5.6-luna")

# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 저장된 벡터 DB 가져와야
load_vs = Chroma(
    collection_name="k_ladder_2026",
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

In [3]:
# 검색기
retriever_mmr = load_vs.as_retriever(search_type="mmr", 
                                 search_kwargs={"k": 5, "fetch_k": 50, "lambda_mult" : 0.25})



In [4]:
# RAG로 붙여보기

SYSTEM_PROMPT = """
너는 공공정책 안내 도우미다.
아래 자료를 참고해서 답해라.
참고 자료에 없으면 "자료에 없음" 이라고 말해라.
정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라
답 끝에 참고한 페이지 번호를 [p.60]과 같은 형식으로 표시해라.
"""

# 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 텍스트로 반환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata["page"]}] \n {doc.page_content} \n\n"

    return context

#### 파이프라인:
- 질문 받아서 rag_docs 만들고
- rag_docs를 format_docs 넣어서 context 만들고
- context를 prompt에 넣어서
- 답변 생성

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# RAG 프롬프트 완성해보기
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ('human', "참고자료\n{context} \n 질문 {question}")
])

# 순서: 먼저 문장이 리트리버 들어가고 formatdocs 통과해 context가 되고, 그와 별개로 문장이 question에 바로 들어가 형성된 뒤 rag_prompt에 또 들어간다!

rag_chain = (# question은 검색기를 거쳐 context라는 key의 value가 돼야 하고, 
    {"context" : (retriever_mmr | format_docs), "question" : RunnablePassthrough()} 
    | rag_prompt    # invoke안의 내용 그대로 prompt에 들어가야
    | model
    | StrOutputParser()
  )  

result = rag_chain.invoke("청년 고용 지원 정책 찾아줘")

In [6]:
print(result)

참고자료에서 확인되는 **청년 고용·창업 지원 정책**은 다음과 같습니다.

1. **청년농업인 영농정착지원사업**
   - 초기 소득 보전 등을 위한 영농정착지원금 지원
   - 최장 3년 지원
   - 신청 기간: 2025년 11월 5일~12월 11일
   - 문의: 청년농업인 콜센터 ☎1670-0255

2. **임대형 스마트팜 조성**
   - 지자체 등이 조성한 스마트팜을 **18~39세 청년농**에게 임대
   - 청년창업보육센터 수료생을 우선 선발
   - 연 1회 신청
   - 김제·상주·고흥·밀양 등 해당 지자체에 신청

3. **스마트팜 청년창업 보육센터**
   - 스마트팜 취·창업을 희망하는 **18~39세 청년** 대상
   - 매년 208명 선발
   - 기초이론부터 경영실습까지 20개월 교육
   - 연 1회 모집공고 시 신청
   - 문의: 한국농업기술진흥원 혁신기술실증팀 ☎063-919-1772

4. **혁신창업 사업화자금(청년전용창업자금)**
   - 대표자가 **39세 이하**이고 업력 3년 미만인 중소기업 또는 창업 예정자
   - 기업당 최대 1억 원, 제조업 및 중점지원분야는 최대 2억 원
   - 대출금리 2.5% 고정
   - 예산 소진 시까지 신청
   - 중소벤처기업진흥공단 누리집에서 온라인 상담 예약 후 신청
   - 문의: 중소기업통합콜센터 ☎1357, 정책자금 콜센터 ☎1811-3655

5. **청년창업사관학교**
   - **39세 이하**, 창업 3년 이내의 창업기업 대표자 대상
   - 창업 인프라·교육·코칭·사업비·기술·연계 지원
   - 2026년 모집 기간: 1월 30일~2월 13일
   - K-Startup 누리집에서 온라인 신청
   - 문의: 중소벤처기업부 ☎1357, 중소벤처기업진흥공단 창업지원처 ☎055-751-9241

6. **학교 밖 청소년 지원**
   - **9세 이상 24세 이하 학교 밖 청소년** 대상
   - 학업 복귀, 진로체험, 직업체험, 진로교육, 취업 지원 활동 제

### 2. 삼성 리포트

In [7]:
from pypdf import PdfReader

reader = PdfReader("../data/Samsung_Electronics_Sustainability_Report_2026_KOR.pdf")
total_k_ladder = len(reader.pages)
total_k_ladder

89

In [8]:
# 임베딩 및 저장

pdf_docs = []

for i,page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_docs.append(Document(page_content=text, 
                             metadata={"source" : "삼성전자_지속가능성_보고서",
                                       "page" : i + 1}
                                       ))

pdf_docs[:5]    

[Document(metadata={'source': '삼성전자_지속가능성_보고서', 'page': 1}, page_content='A Journey  Towards\na Sustainable Future 2026\n삼성전자 지속가능경영보고서\nA Journey Towards\na Sustainable Future'),
 Document(metadata={'source': '삼성전자_지속가능성_보고서', 'page': 2}, page_content='삼성전자 지속가능경영보고서 2026 2Our Company AppendixFacts & Figures PrinciplePlanet People\nA Journey  Towards\na Sustainable Future\nA Journey Towards\na Sustainable Future\n삼성전자 지속가능경영보고서 2026\nCEO 메시지\n회사소개\n기업 지배구조\n이해관계자 소통\n분야별 주요성과 \n[DX부문] \n추진체계와 주요성과 \n기후변화 \n자원순환\n수자원\n오염물질\n생물다양성\n경제성과\n사회성과\n환경성과 \n사업부문별 환경성과 \n[DS부문] \n추진체계와 주요성과 \n기후변화 \n자원순환\n수자원\n오염물질\n생물다양성\n중대성 평가\n독립된 인증인의 인증보고서\nScope 1, 2 온실가스 배출량 검증 의견서 \nScope 3 온실가스 배출량 검증 의견서 \nGRI Index\nTCFD 대조표\nSASB 대조표\nAbout This Report \n임직원\n공급망\n사회공헌\n개인정보보호와 보안\nAI윤리\n제품 품질과 안전\nOur Company Planet\nFacts & Figures Appendix \nPeople\n04\n05\n06\n07\n08\n11\n12\n15\n17\n19\n20\n64\n65\n69\n72\n22\n23\n27\n29\n31\n33\n10\n76\n78\n79\n80\n82\n84\n86\n88\n36\n45\n51\n53\n55\n57\n준법과 

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, # 사이즈 기준으로 자동 자름
                                          chunk_overlap = 80)   # 일부러 겹치는 부분 있도록

chunks = splitter.split_documents(pdf_docs)
len(chunks)

610

In [10]:
# 임베딩 및 저장
DB_PATH_Samsung = "../data/Samsung"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터스토어에 저장
vectorstore = Chroma.from_documents(
    documents=chunks,  
    embedding=embeddings,
    collection_name="Samsung_Sustainability",
    persist_directory=DB_PATH_Samsung
)

In [11]:
retriever_mmr = vectorstore.as_retriever(search_type="mmr", 
                                 search_kwargs={"k": 5, "fetch_k": 50, "lamda_mult" : 0.25})

In [12]:
SYSTEM_PROMPT_SAMSUNG = """
너는 삼성전자 정보 답변 도우미다.
아래 자료를 참고해서 답해라.
참고 자료에 없으면 "자료에 없음" 이라고 말해라.
정확한 정보는 매뉴얼을 직접 확인하라고 말해라.
답 끝에 참고한 페이지 번호를 [p.60]과 같은 형식으로 표시해라.
"""

In [13]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_SAMSUNG),
    ('human', "참고자료\n{context} \n 질문 {question}")
])

# 순서: 먼저 문장이 리트리버 들어가고 formatdocs 통과해 context가 되고, 그와 별개로 문장이 question에 바로 들어가 형성된 뒤 rag_prompt에 또 들어간다!

rag_chain = (# question은 검색기를 거쳐 context라는 key의 value가 돼야 하고, 
    {"context" : (retriever_mmr | format_docs), "question" : RunnablePassthrough()} 
    | rag_prompt    # invoke안의 내용 그대로 prompt에 들어가야
    | model
    | StrOutputParser()
  )  

result = rag_chain.invoke("수질오염 다중방어체계 설명해줘")

In [14]:
print(result)

수질오염 다중방어체계는 사업장에서 발생하는 폐수가 환경에 미치는 영향을 줄이기 위해 여러 단계의 관리·처리 절차를 적용하는 체계입니다.

주요 내용은 다음과 같습니다.

1. **오염물질별 법규 및 기준 관리**  
   수질오염물질 배출 관련 법적 기준을 준수하도록 내부 관리 기준을 수립하고 적용합니다.

2. **사용 전 관리**  
   사업장에서 사용하는 화학물질에 대해 사내 반입 전 의뢰 및 검토를 실시하고, 물질의 성상에 따라 분리 관리합니다.

3. **배출 모니터링 및 누출 예방**  
   배출 상태를 모니터링하고, 교육과 누출 예방 활동 등을 통해 오염 발생을 사전에 줄입니다.

4. **다단계 폐수처리**  
   최적방지기술을 적용하고 폐수처리 과정을 **4단계로 구축·운영**하여 수질오염물질 배출을 최소화합니다.

5. **처리·배출 기준 준수 확인**  
   관련 규정과 사내 기준의 충족도를 검토한 뒤, 폐수를 안전하게 배출·처리·폐기합니다.

다만 자료에는 4단계 폐수처리의 **각 단계별 명칭과 세부 처리 방식은 나와 있지 않습니다.** 정확한 단계 구성은 해당 환경·폐수처리 매뉴얼을 직접 확인해야 합니다. [p.31]


### 3. 개인정보 FAQ

In [15]:
import pandas as pd

df = pd.read_csv("../data/16-6_개인정보FAQ.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   처리상황단계내용  100 non-null    str  
 1   적용분야내용    100 non-null    str  
 2   개인정보유형내용  100 non-null    str  
 3   코드제목      100 non-null    str  
 4   주제내용      100 non-null    str  
 5   문제상황내용    100 non-null    str  
 6   질문        100 non-null    str  
 7   해결방법내용    100 non-null    str  
 8   결론내용      100 non-null    str  
 9   유사사례내용    85 non-null     str  
 10  faq식별자    100 non-null    str  
 11  최종수정년월일시  100 non-null    str  
 12  조회수       100 non-null    int64
 13  임시        100 non-null    str  
dtypes: int64(1), str(13)
memory usage: 11.1 KB


In [16]:
df.index[0]

0

In [17]:
df.head(3)

,처리상황단계내용,적용분야내용,개인정보유형내용,코드제목,주제내용,문제상황내용,질문,해결방법내용,결론내용,유사사례내용,faq식별자,최종수정년월일시,조회수,임시
0,제공,금융 분야,일반정보,채무보증인의 상속인에게 채무자의 개인정보 제공 가능?,채무보증인의 상속인에게 원 채무자의 개인정보 제공 정당성,"A는 직무수행 중 부상을 당해 전역하였으며, 전역 후 사업을 하기위해 국가유공자 신...","상속인의 요청에 따라 은행은 채무자 A의 연락처(변경 전 전화번호, 우편물 반송된 ...",개인정보보호법에 따라 개인정보처리자(은행)는 정보주체 또는 제3자의 이익을 부당하게...,은행은 이미 상당한 노력을 기울였음에도 불구하고 채무자 A와의 연락이 두절되어 A로...,"FAQ_0000000000000714,FAQ_0000000000000751",FAQ_0000000000000811,2013-10-14 15:41:27,6003,제공
1,수집·이용,공공기관,일반정보,행정기관 간의 민원 이관 시 개인정보도 함께 제공된다?,민원을 이관한 행정기관의 민원인의 개인정보 이용,"제가 A지방자치단체에 민원을 제기하면서 이름, 전화번호, 주소 등을 알려주었는데 민...",A기관은 민원인의 동의 없이 개인정보를 B기관에 제공할 수 있는 건지요?,개인정보 보호법은 개인정보 보호에 관하여 다른 법률에 특별한 규정이 있는 경우 해당...,민원사무처리에 관한 법률에 따라 A기관은 접수된 민원이 B기관 소관인 경우 민원인의...,"FAQ_0000000000000717,FAQ_0000000000000730,FAQ_...",FAQ_0000000000000726,2013-10-07 11:45:18,3098,수집·이용
2,수집·이용,인사업무 분야,인사정보,회사 입사지원자에게도 개인정보를 요구한다?,회사 입사지원자의 개인정보 수집,A회사는 직원 채용을 위하여 입사지원자의 개인정보를 수집하려고 합니다. 개인정보를 ...,기업이 입사지원자의 개인정보를 수집하고자 할 때 개인정보의 수집동의서를 별도로 받아...,계약 체결 및 이행을 위하여 불가피하게 필요한 경우 동의 없이 개인정보를 수집할 수...,입사지원은 근로계약 체결의 일부로 입사지원자의 동의 없이 채용에 필요한 최소한의 정...,"FAQ_0000000000000744,FAQ_0000000000000772,FAQ_...",FAQ_0000000000000734,2013-10-11 18:48:49,10609,수집·이용


In [29]:
docs = []

for i in df.index:
    text = df['질문'].iloc[i]
    docs.append(Document(page_content=text, 
                             metadata={"source" : "개인정보FAQ",
                                       "질문번호" : i+1,
                                       "적용분야내용" : df['적용분야내용'].iloc[i],
                                       "개인정보유형내용" : df['개인정보유형내용'].iloc[i],
                                       "코드제목" : df['코드제목'].iloc[i],
                                       "주제내용" : df['주제내용'].iloc[i],
                                       "문제상황내용" : df['문제상황내용'].iloc[i],
                                       "해결방법내용" : df['해결방법내용'].iloc[i],
                                       "결론내용" : df['결론내용'].iloc[i]}
                                       ))

docs[:5]

[Document(metadata={'source': '개인정보FAQ', '질문번호': 1, '적용분야내용': '금융 분야', '개인정보유형내용': '일반정보', '코드제목': '채무보증인의 상속인에게 채무자의 개인정보 제공 가능?', '주제내용': '채무보증인의 상속인에게 원 채무자의 개인정보 제공 정당성', '문제상황내용': 'A는 직무수행 중 부상을 당해 전역하였으며, 전역 후 사업을 하기위해 국가유공자 신분으로 B를 보증인으로 하여 은행으로부터 유리한 조건으로 대출을 받았습니다. A는 어느 날부터 원리금을 미납하기 시작하여 은행은 전화 및 우편 등을 통해 A에게 연락을 취하였으나 전화번호는 변경되었고, 발송된 우편물은 수신자 미거주로 반송되었습니다. 이후 A의 보증인 B마저 사망하자 은행은 B의 상속인에게 A의 대부원리금 상환을 요구하며 부동산과 예금에 가압류 조치를 할 수 있음을 통보하였습니다. B의 상속인은 대부원리금 상환 요구에 대한 억울함을 호소하며 은행에게 채무자 A의 연락처를 알려달라고 합니다.', '해결방법내용': '개인정보보호법에 따라 개인정보처리자(은행)는 정보주체 또는 제3자의 이익을 부당하게 침해할 우려가 있을 때를 제외하고 정보주체 또는 그 법정대리인이 주소불명 등으로 사전 동의를 받을 수 없는 경우로, 명백히 정보주체 또는 제3자의 급박한 생명, 신체, 재산의 이익을 위하여 필요하다고 인정되는 경우에는 개인정보를 당초 수집 목적 외의 용도로 이용하거나 제3자에게 제공할 수 있습니다.', '결론내용': '은행은 이미 상당한 노력을 기울였음에도 불구하고 채무자 A와의 연락이 두절되어 A로부터 사전 동의를 받을 수 없으며, 대부원리금 변제와 관련한 상속인의 급박한 재산상 이익을 위하여 필요하다고 인정되므로 채무자 A의 동의 없이 연락처를 상속인에게 제공할 수 있습니다.'}, page_content='상속인의 요청에 따라 은행은 채무자 A의 연락처(변경 전 전화번호, 우편물 반송된 주소 등을 말함)를 상속인에게 제공할 수 있는지요?'),

In [30]:
# 임베딩 및 저장
DB_PATH_FAQ = "../data/FAQ"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터스토어에 저장
vectorstore_FAQ = Chroma.from_documents(
    documents=docs,  
    embedding=embeddings,
    collection_name="FAQ",
    persist_directory=DB_PATH_FAQ
)

In [31]:
docs[0].metadata['질문번호']

1

In [32]:
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata["질문번호"]}] \n 질문: {doc.page_content} \n 답변: {doc.metadata["결론내용"]}\n\n"

    return context

In [33]:
retriever_mmr = vectorstore_FAQ.as_retriever(search_type="mmr", 
                                 search_kwargs={"k": 5, "fetch_k": 50, "lambda_mult" : 0.25})

SYSTEM_PROMPT_FAQ = """
너는 FAQ 정보 답변 도우미다.
아래 자료를 참고해서 답해라.
참고 자료에 없으면 "자료에 없음" 이라고 말해라.
정확한 정보는 원본 데이터프레임을 직접 확인하라고 말해라.
답 끝에 참고한 질문 번호를 [질문번호 60]과 같은 형식으로 표시해라.
"""

In [34]:
# retriever_mmr = retriever_mmr.invoke("cctv")
# retriever_mmr

In [35]:
df['질문'].iloc[80]

'시설안전, 화재예방 및 범죄예방의 목적으로 설치하여 수집한 영상정보를 근로자의 근태관리를 위해 이용할 수 있는지요?'

In [36]:

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_FAQ),
    ('human', "참고자료\n{context} \n 질문 {question}")
])

# 순서: 먼저 문장이 리트리버 들어가고 formatdocs 통과해 context가 되고, 그와 별개로 문장이 question에 바로 들어가 형성된 뒤 rag_prompt에 또 들어간다!

rag_chain = (# question은 검색기를 거쳐 context라는 key의 value가 돼야 하고, 
    {"context" : (retriever_mmr | format_docs), "question" : RunnablePassthrough()} 
    | rag_prompt    # invoke안의 내용 그대로 prompt에 들어가야
    | model
    | StrOutputParser()
  )  

result = rag_chain.invoke("영상정보를 근로자의 근태관리를 위해 이용할 수 있는지요?")

In [37]:
print(result)

시설안전·화재예방·범죄예방 목적으로 설치한 CCTV 영상정보는 해당 목적에 한해 이용해야 하므로, 근로자의 근태관리나 부정행위 감시 목적으로 이용할 수 없습니다. 정확한 정보는 원본 데이터프레임을 직접 확인하세요. [질문번호 81]
